# Overlay Histograms - Distances between endpoints & branching points to bone outside

Input: distance transformation map from the "09_BP_EP_distances.ijm" Fiji script

Output: Histogram of distance distribution (in um) as txt-file

In [ ]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage import io

## Define standard colors and legend titles

In [ ]:
# Young mice without blood loss => shades of blue
Ycolors = ['lightblue', 'deepskyblue', 'dodgerblue', 'blue', 'navy']
Ylabels = ['VK-AA498', 'VK-AA499', 'VK-AA500', 'VK-AA501', 'VK-AA502']

# Young. blood loss mice => shades of orange
Bcolors = ['peachpuff', 'sandybrown', 'darkorange', 'chocolate', 'saddlebrown']
Blabels = ['VK-AA763', 'VK-AA764', 'VK-AA765', 'VK-AA766', 'VK-AA767']

# Old mice => shades of grey
Ocolors = ['black', 'dimgrey', 'grey', 'darkgrey', 'gainsboro']
Olabels = ['VK-AA758', 'VK-AA759', 'VK-AA760', 'VK-AA761', 'VK-AA762']

## Define datasets

In [ ]:
# Young mice 
Ymouse_1 = './Histograms/VK-AA498_shaft_679-1358_BP_dist.tif'
Ymouse_2 = './Histograms/VK-AA499_shaft_590-1180_BP_dist.tif'
Ymouse_3 = './Histograms/VK-AA500_shaft_716-1432_BP_dist.tif'
Ymouse_4 = './Histograms/VK-AA501_shaft_679-1358_BP_dist.tif'
Ymouse_5 = './Histograms/VK-AA502_shaft_669-1338_BP_dist.tif'

# Load also the base skeletonized points file to get base number of zeros
Ymouse_1_scaled = 'VK-AA763_shaft_612-1224_BP_scaled.tif'
Ymouse_2_scaled = 'VK-AA764_shaft_649-1298_BP_scaled.tif'
Ymouse_3_scaled = 'VK-AA765_shaft_630-1260_BP_scaled.tif'
Ymouse_4_scaled = 'VK-AA766_shaft_599-1198_BP_scaled.tif'
Ymouse_5_scaled = 'VK-AA767_shaft_619-1238_BP_scaled.tif'


In [ ]:
# Young, blood-loss mice 
Bmouse_1 = './Histograms/VK-AA763_shaft_612-1224_BP_dist.tif'
Bmouse_2 = './Histograms/VK-AA764_shaft_649-1298_BP_dist.tif'
Bmouse_3 = './Histograms/VK-AA765_shaft_630-1260_BP_dist.tif'
Bmouse_4 = './Histograms/VK-AA766_shaft_599-1198_BP_dist.tif'
Bmouse_5 = './Histograms/VK-AA767_shaft_619-1238_BP_dist.tif'

# Load also the base skeletonized points file to get base number of zeros
Bmouse_1_scaled = 'VK-AA758_shaft_783-1566_BP_scaled.tif'
Bmouse_2_scaled = 'VK-AA759_shaft_778-1556_BP_scaled.tif'
Bmouse_3_scaled = 'VK-AA760_shaft_819-1638_BP_scaled.tif'
Bmouse_4_scaled = 'VK-AA761_shaft_767-1534_BP_scaled.tif'
Bmouse_5_scaled = 'VK-AA762_shaft_830-1660_BP_scaled.tif'


In [ ]:
# Aged mice 
Omouse_1 = './Histograms/VK-AA758_shaft_783-1566_BP_dist.tif'
Omouse_2 = './Histograms/VK-AA759_shaft_778-1556_BP_dist.tif'
Omouse_3 = './Histograms/VK-AA760_shaft_819-1638_BP_dist.tif'
Omouse_4 = './Histograms/VK-AA761_shaft_767-1534_BP_dist.tif'
Omouse_5 = './Histograms/VK-AA762_shaft_830-1660_BP_dist.tif'

# Load also the base skeletonized points file to get base number of zeros
Omouse_1_scaled = 'VK-AA758_shaft_391-1174_BP_scaled.tif'
Omouse_2_scaled = 'VK-AA759_shaft_389-1167_BP_scaled.tif'
Omouse_3_scaled = 'VK-AA760_shaft_409-1228_BP_scaled.tif'
Omouse_4_scaled = 'VK-AA761_shaft_383-1150_BP_scaled.tif'
Omouse_5_scaled = 'VK-AA762_shaft_415-1245_BP_scaled.tif'


## Define function to plot the overlaid density distribution

In [ ]:
# Function to plot the density histograms all in one plot using numpy bincount
def plot_using_bincount(datasets, colors=None, labels=None, title="Histogram", xlabel="distance [µm]", ylabel="Frequency"):
    plt.figure(figsize=(10, 6))
    
    bin_indices = np.arange(1, 251) # attention: distance map is given in pixel
    bin_indices_um = bin_indices * 2.6 # convert to um
    
    for i, data in enumerate(datasets):
        # Exclude zeros and limit to the lowest 500 intensity bins
        data = data[(data > 0) & (data < 250)]
        
        # Count occurrences in each bin using np.bincount and normalize to number of occurrences
        bin_counts = np.bincount(data, minlength=250)[:250] / data.size
        
        # Plot using np.arange for x-axis (bin edges) and np.bincount for y-axis (frequencies)
        plt.plot(bin_indices_um, bin_counts, color=colors[i] if colors else None, label=labels[i] if labels else None, lw=2)

    # Add labels and title
    plt.xlabel(xlabel, fontsize=14, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.title(title, fontsize=16, fontweight='bold')

    # Add legend if labels are provided
    if labels:
        plt.legend()

    # Show the plot
    plt.show()

# Function to save the density histograms
def save_density_histograms_to_csv(datasets, csv_filename="combined_density_histograms.csv"):
    
    # Create an empty list to store histogram dataframes
    dfs = []
    
    for i, data in enumerate(datasets):
        # Exclude zeros and limit to the lowest 500 intensity bins
        data = data[(data > 0) & (data < 250)]
        
        # Compute density histogram values
        bin_indices = np.arange(0, 250)
        bin_indices_um = bin_indices * 2.6  # Convert to µm
        values = np.bincount(data, minlength=250)[:250]
        
        # Create a DataFrame for the current dataset's histogram
        df = pd.DataFrame({'Bin Edges': bin_indices_um, f'Density Values_{i}': values})  # Add a column with density values for the current datasets
        
        # Append the DataFrame to the list
        dfs.append(df)
        
    # Merge all the individual dataframes on the 'Bin Edges' column
    merged_df = dfs[0]  # Start with the first DataFrame
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on='Bin Edges', how='outer')
    
    # Save the merged DataFrame to the specified CSV file
    merged_df.to_csv(csv_filename, index=False)


# Remove background zeros and replace with -1
def compare_and_replace_zeros(list1, list2):

    # Check if both lists have the same length
    if len(list1) != len(list2):
        raise ValueError("Lists must have the same length for element-wise comparison.")

    # Iterate through the lists and compare/replace for each corresponding pair
    for i in range(len(list1)):
        array1 = list1[i]
        array2 = list2[i]

        # Element-wise comparison and replacement
        zero_mask = (array1 == 0) & (array2 == 0)
        array1[zero_mask] = -1
        array2[zero_mask] = -1

    return list1, list2

## Young mice dataset

In [ ]:
# Load young mice data sets
Ydatasets = [Ymouse_1, Ymouse_2, Ymouse_3, Ymouse_4, Ymouse_5]
Ydatasets_outlines = [Ymouse_1_scaled, Ymouse_2_scaled, Ymouse_3_scaled, Ymouse_4_scaled, Ymouse_5_scaled]

Y_outlines = list()
Yall_distance= list()

for dataset in Ydatasets:
    img = io.imread(dataset)
    Yall_distance.append(img)

for dataset in Ydatasets_outlines:
    img = io.imread(dataset)
    Y_outlines.append(img)

cleaned_Yall_distance, _ = compare_and_replace_zeros(Yall_distance, Y_outlines)
# Note: only the peak region is plotted (from 0 -50 µm)
plot_using_bincount(cleaned_Yall_distance, colors=Ycolors, labels=Ylabels, title='Young Mice')

## Young, blood loss mice dataset

In [ ]:
# Load young, blood loss mice data sets
Bdatasets = [Bmouse_1, Bmouse_2, Bmouse_3, Bmouse_4, Bmouse_5]
Bdatasets_outlines = [Bmouse_1_scaled, Bmouse_2_scaled, Bmouse_3_scaled, Bmouse_4_scaled, Bmouse_5_scaled]

B_outlines = list()
Ball_distance = list()

for dataset in Bdatasets:
    img = io.imread(dataset)
    Ball_distance.append(img)

for dataset in Bdatasets_outlines:
    img = io.imread(dataset)
    B_outlines.append(img)

cleaned_Ball_distance, _ = compare_and_replace_zeros(Ball_distance, B_outlines)
# Note: only the peak region is plotted (from 0 -50 µm)
plot_using_bincount(cleaned_Ball_distance, colors=Bcolors, labels=Blabels, title='Young, blood loss Mice')

## Aged mice

In [ ]:
# Load young, blood loss mice data sets
Odatasets = [Omouse_1, Omouse_2, Omouse_3, Omouse_4, Omouse_5]
Odatasets_scaled = [Omouse_1_scaled, Omouse_2_scaled, Omouse_3_scaled, Omouse_4_scaled, Omouse_5_scaled]

O_outlines = list()
Oall_distance = list()

for dataset in Odatasets:
    img = io.imread(dataset)
    Oall_distance.append(img)

for dataset in Odatasets_scaled:
    img = io.imread(dataset)
    O_outlines.append(img)

cleaned_Oall_distance, _ = compare_and_replace_zeros(Oall_distance, O_outlines)
# Note: only the peak region is plotted (from 0 -50 µm)
plot_using_bincount(cleaned_Oall_distance, colors=Ocolors, labels=Olabels, title='Aged Mice')

## Save histograms (density)

In [ ]:
save_density_histograms_to_csv(cleaned_Ball_distance, csv_filename="Blood-let_BP_3.csv")
save_density_histograms_to_csv(cleaned_Yall_distance, csv_filename="Young_BP_3.csv")
save_density_histograms_to_csv(cleaned_Oall_distance, csv_filename="Old_BP_3.csv")